In [ ]:
%pip install git+https://github.com/Eickland/HumSpectra.git --upgrade

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path
from pandas import DataFrame
from typing import List
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
from io import StringIO
import sys
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, 
                             mean_squared_error, r2_score, mean_absolute_error)
warnings.filterwarnings('ignore')
matplotlib.use('Agg')
%matplotlib inline

import HumSpectra.fluorescence as fl
import HumSpectra.ultraviolet as uv 
import HumSpectra.utilits as ut

In [160]:
wdir_date = Path.cwd().parent / "Initial data" / "2025"
fluo_path = Path.joinpath(wdir_date, "Fluorescence")
uv_path = Path.joinpath(wdir_date, "UV","Individual spectra")
wdir_savepath = Path.joinpath(Path.cwd().parent,"Server","Received data","Clustering")

carbon_data = pd.read_excel(Path.joinpath(wdir_date, 'C_org.xlsx'))
geodata = pd.read_excel(Path.joinpath(wdir_date, 'GeoData.xlsx'))

fluo_spectra = ut.load_spectra_data(fluo_path, fl.read_fluo_3d, index_col=1)
uv_spectra = ut.load_spectra_data(uv_path, uv.read_csv_uv, spectra_type="absorption")

parafac_data = pd.read_csv(Path.joinpath(Path.cwd().parent,"Server","Received data","Parafac","parafac_2025.csv"))
parafac_data.rename(columns={parafac_data.columns[0]:"Sample"},inplace=True)

fl_data = ut.spectra_to_df(fluo_spectra, {
    "Asm_350": fl.asm_350, "Asm_280": fl.asm_280,
    "FIX": fl.fluo_index, "HIX": fl.humin_index
})

uv_data = ut.spectra_to_df(uv_spectra, {
    "E2E3": uv.e2_e3, "epsilon": uv.epsilon
})

MainData_init = (uv_data.merge(carbon_data, on="Sample")
                .assign(Suva=lambda x: 100*x.epsilon / x.C_org)
                .drop(columns=["epsilon"])
                .merge(geodata[["Sample", "pH", "eH"]], on="Sample")
                .merge(fl_data, on=["Sample", "Class", "Subclass"])
                .merge(parafac_data,on="Sample"))

numeric_cols = MainData_init.select_dtypes(include=['float']).columns
MainData_init[numeric_cols] = MainData_init[numeric_cols].map(lambda x: float(f"{x:.4g}"))
MainData_init = MainData_init[MainData_init["Subclass"].isin(['Storage 2', 'Storage 3', 'Storage 4', 'Storage 5', 'Storage 6', 'Storage 7'])]
MainData_init.Sample = MainData_init.Sample.apply(lambda x: x+"-2025")
#MainData_init.set_index(["Class","Subclass","Sample"],inplace=True)

#storage_list = MainData_init.index.get_level_values(1).unique().to_list()
feature_columns = ["E2E3","C_org","Suva","Asm_350","pH","Asm_280","Component_1","Component_2","Component_3"]

MainData_2025 = ut.delete_eject_iqr(MainData_init,iqr_param=2.3,multi_index=False,columns=["Suva","Asm_280","C_org"])

In [161]:
wdir_date = Path.cwd().parent / "Initial data" / "2024"
fluo_path = Path.joinpath(wdir_date, "Fluorescence")
uv_path = Path.joinpath(wdir_date, "UV","Individual spectra")
wdir_savepath = Path.joinpath(Path.cwd().parent,"Server","Received data","Clustering")

carbon_data = pd.read_excel(Path.joinpath(wdir_date, 'C_org.xlsx'))
geodata = pd.read_excel(Path.joinpath(wdir_date, 'GeoData.xlsx'))

fluo_spectra = ut.load_spectra_data(fluo_path, fl.read_fluo_3d, index_col=0)
uv_spectra = ut.load_spectra_data(uv_path, uv.read_csv_uv, spectra_type="absorption")

parafac_data = pd.read_csv(Path.joinpath(Path.cwd().parent,"Server","Received data","Parafac","parafac_2024.csv"))
parafac_data.rename(columns={parafac_data.columns[0]:"Sample"},inplace=True)

fl_data = ut.spectra_to_df(fluo_spectra, {
    "Asm_350": fl.asm_350, "Asm_280": fl.asm_280,
    "FIX": fl.fluo_index, "HIX": fl.humin_index
})

uv_data = ut.spectra_to_df(uv_spectra, {
    "E2E3": uv.e2_e3, "epsilon": uv.epsilon
})

MainData_init = (uv_data.merge(carbon_data, on="Sample")
                .assign(Suva=lambda x: 100*x.epsilon / x.C_org)
                .drop(columns=["epsilon"])
                .merge(geodata[["Sample", "pH", "eH"]], on="Sample")
                .merge(fl_data, on=["Sample", "Class", "Subclass"])
                .merge(parafac_data,on="Sample"))

numeric_cols = MainData_init.select_dtypes(include=['float']).columns
MainData_init[numeric_cols] = MainData_init[numeric_cols].map(lambda x: float(f"{x:.4g}"))
MainData_init = MainData_init[MainData_init["Subclass"].isin(['Storage 2', 'Storage 3', 'Storage 4', 'Storage 5', 'Storage 6', 'Storage 7'])]
MainData_init.Sample = MainData_init.Sample.apply(lambda x: x+"-2024")
#MainData_init.set_index(["Class","Subclass","Sample"],inplace=True)

#storage_list = MainData_init.index.get_level_values(1).unique().to_list()
feature_columns = ["E2E3","C_org","Suva","Asm_350","pH","Asm_280","Component_1","Component_2","Component_3"]

MainData_2024 = ut.delete_eject_iqr(MainData_init,iqr_param=1.5,multi_index=False,columns=["Suva","Asm_280","C_org"])

In [ ]:
MainData = pd.concat([MainData_2024,MainData_2025])
MainData.reset_index(drop=True)

,Sample,Class,Subclass,E2E3,C_org,Suva,pH,eH,Asm_350,Asm_280,FIX,HIX,Component_1,Component_2,Component_3
0,ADOM-SL2-11-2024,ADOM,Storage 2,3.520,22.910,4.383,5.60,80.0,4.017,0.5249,1.261,24.110,2.776,82.660,14.57
1,ADOM-SL2-3-2024,ADOM,Storage 2,3.416,25.230,3.908,5.50,114.0,4.103,0.7317,1.241,19.050,4.282,79.510,16.21
2,ADOM-SL2-4-2024,ADOM,Storage 2,3.421,16.210,2.677,4.90,45.0,4.461,0.9186,1.272,15.070,5.410,74.300,20.29
3,ADOM-SL2-5-2024,ADOM,Storage 2,3.449,17.720,2.617,5.30,140.0,4.007,0.4940,1.216,27.430,2.623,84.620,12.75
4,ADOM-SL2-6-2024,ADOM,Storage 2,3.819,13.710,2.646,6.00,106.0,4.407,0.7856,1.289,17.410,5.189,77.250,17.56
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101,ADOM-SL7-5-2025,ADOM,Storage 7,3.800,7.129,1.459,8.21,12.0,4.005,1.0120,1.342,7.656,30.480,6.095,63.43
102,ADOM-SL7-6-2025,ADOM,Storage 7,3.952,9.097,1.000,8.64,100.0,3.690,0.9691,1.295,8.146,30.310,5.986,63.70
103,ADOM-SL7-7-2025,ADOM,Storage 7,4.000,6.717,1.533,8.72,90.0,4.220,1.0220,1.391,7.665,30.630,5.936,63.43
104,ADOM-SL7-8-2025,ADOM,Storage 7,3.741,7.297,1.549,8.67,80.0,3.882,0.8738,1.314,9.354,33.390,6.508,60.10


In [23]:
def cluster_analysis_with_subclasses(df, n_clusters=None, random_state=42, output_html_path=None):
    """
    Кластеризация данных с многоуровневым индексом и анализ подклассов
    с выводом результатов в HTML файл
    
    Parameters:
    df - DataFrame с многоуровневым индексом
    n_clusters - количество кластеров (если None, подбирается автоматически)
    random_state - для воспроизводимости
    output_html_path - путь для сохранения HTML отчета
    """
    
    # Перехватываем вывод в консоль
    old_stdout = sys.stdout
    sys.stdout = captured_output = StringIO()
    
    try:
        # Сохраняем оригинальные индексы
        original_index = df.index
        subclasses = df.index.get_level_values(1).unique()  # Все уникальные подклассы
        
        print("=" * 60)
        print("АНАЛИЗ КЛАСТЕРИЗАЦИИ С ПОДКЛАССАМИ")
        print("=" * 60)
        
        # Шаг 1: Предобработка данных
        print("1. Предобработка данных...")
        
        # Удаляем нечисловые колонки если есть
        numeric_columns = df.select_dtypes(include=[np.number]).columns
        df_numeric = df[numeric_columns].copy()
        
        # Проверяем наличие пропущенных значений
        if df_numeric.isnull().sum().sum() > 0:
            print(f"   Обнаружено пропущенных значений: {df_numeric.isnull().sum().sum()}")
            df_numeric = df_numeric.fillna(df_numeric.mean())
        
        # Масштабирование признаков
        scaler = StandardScaler()
        data_scaled = scaler.fit_transform(df_numeric)
        
        print(f"   Размерность данных: {data_scaled.shape}")
        print(f"   Количество признаков: {data_scaled.shape[1]}")
        
        # Шаг 2: Определение оптимального числа кластеров (если не задано)
        if n_clusters is None:
            print("2. Подбор оптимального числа кластеров...")
            best_k = 2
            best_silhouette = -1
            
            for k in range(2, min(11, len(df) // 2)):
                kmeans = KMeans(n_clusters=k, random_state=random_state, n_init=10)
                cluster_labels = kmeans.fit_predict(data_scaled)
                silhouette_avg = silhouette_score(data_scaled, cluster_labels)
                
                print(f"   k={k}: Silhouette Score = {silhouette_avg:.4f}")
                
                if silhouette_avg > best_silhouette:
                    best_silhouette = silhouette_avg
                    best_k = k
            
            n_clusters = best_k
            print(f"   Оптимальное число кластеров: {n_clusters} (silhouette: {best_silhouette:.4f})")
        else:
            print(f"2. Используется заданное число кластеров: {n_clusters}")
        
        # Шаг 3: Кластеризация K-Means
        print("3. Выполнение кластеризации K-Means...")
        kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
        cluster_labels = kmeans.fit_predict(data_scaled)
        
        # Добавляем метки кластеров к данным
        result_df = df.copy()
        result_df['cluster'] = cluster_labels
        
        # Шаг 4: Анализ распределения подклассов по кластерам
        print("4. Анализ распределения подклассов по кластерам...")
        
        # Создаем таблицу сопряженности: подклассы vs кластеры
        contingency_table = pd.crosstab(
            index=result_df.index.get_level_values(1),  # подклассы
            columns=result_df['cluster'],
            margins=True,
            margins_name='Всего'
        )
        
        # Процентное распределение по строкам (какой % подкласса попал в каждый кластер)
        percentage_table = pd.crosstab(
            index=result_df.index.get_level_values(1),
            columns=result_df['cluster'],
            normalize='index'
        ) * 100
        
        # Шаг 5: Вывод результатов
        print("\n" + "=" * 60)
        print("РЕЗУЛЬТАТЫ КЛАСТЕРИЗАЦИИ")
        print("=" * 60)
        
        print(f"\nОбщая информация:")
        print(f"- Всего наблюдений: {len(result_df)}")
        print(f"- Количество кластеров: {n_clusters}")
        print(f"- Количество подклассов: {len(subclasses)}")
        print(f"- Silhouette Score: {silhouette_score(data_scaled, cluster_labels):.4f}")
        
        print(f"\nРаспределение по кластерам:")
        cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()
        for cluster, count in cluster_counts.items():
            print(f"  Кластер {cluster}: {count} наблюдений ({count/len(result_df)*100:.1f}%)")
        
        print(f"\nТАБЛИЦА СОПРЯЖЕННОСТИ: Подклассы × Кластеры")
        print("-" * 50)
        print(contingency_table)
        
        print(f"\nПРОЦЕНТНОЕ РАСПРЕДЕЛЕНИЕ (% по строкам)")
        print("-" * 50)
        print(percentage_table.round(1))
        
        # Анализ "чистоты" кластеров
        print(f"\nАНАЛИЗ 'ЧИСТОТЫ' КЛАСТЕРОВ:")
        print("-" * 50)
        
        cluster_summary = []
        for cluster in range(n_clusters):
            cluster_data = result_df[result_df['cluster'] == cluster]
            cluster_subclasses = cluster_data.index.get_level_values(1)
            
            if len(cluster_data) > 0:
                # Самый частый подкласс в кластере
                subclass_counts = cluster_subclasses.value_counts()
                top_subclass = subclass_counts.index[0]
                top_count = subclass_counts.iloc[0]
                top_percentage = (top_count / len(cluster_data)) * 100
                
                # Количество уникальных подклассов в кластере
                unique_subclasses = cluster_subclasses.nunique()
                
                cluster_summary.append({
                    'Кластер': cluster,
                    'Наблюдений': len(cluster_data),
                    'Уникальных подклассов': unique_subclasses,
                    'Доминирующий подкласс': top_subclass,
                    'Доля доминирующего': f"{top_percentage:.1f}%"
                })
        
        cluster_summary_df = pd.DataFrame(cluster_summary)
        print(cluster_summary_df.to_string(index=False))
        
        # Дополнительный анализ характеристик кластеров
        print(f"\nХАРАКТЕРИСТИКИ КЛАСТЕРОВ (средние значения):")
        print("-" * 60)
        
        numeric_columns = result_df.select_dtypes(include=[np.number]).columns
        numeric_columns = numeric_columns[numeric_columns != 'cluster']  # Исключаем колонку с кластерами
        
        cluster_means = result_df.groupby('cluster')[numeric_columns].mean()
        print(cluster_means.round(3))
        
        # Маппинг подклассов на основные кластеры
        print(f"\nМАППИНГ ПОДКЛАССОВ НА ОСНОВНЫЕ КЛАСТЕРЫ:")
        print("-" * 50)
        
        subclass_cluster_map = {}
        for subclass in result_df.index.get_level_values(1).unique():
            subclass_data = result_df.xs(subclass, level=1)
            if len(subclass_data) > 0:
                # Определяем основной кластер для подкласса
                main_cluster = subclass_data['cluster'].mode()
                if len(main_cluster) > 0:
                    subclass_cluster_map[subclass] = main_cluster[0]
                    print(f"  {subclass} → Кластер {main_cluster[0]}")
        
        # Получаем весь вывод
        console_output = captured_output.getvalue()
        
        # Восстанавливаем stdout
        sys.stdout = old_stdout
        
        # Создаем HTML отчет
        create_html_report(console_output, result_df, contingency_table, 
                          percentage_table, cluster_summary_df, cluster_means, 
                          subclass_cluster_map, n_clusters, output_html_path)
        
        print(f"\n✅ HTML отчет сохранен в файл: {output_html_path}")
        
        return result_df, kmeans, scaler, contingency_table, percentage_table, subclass_cluster_map
        
    except Exception as e:
        # Восстанавливаем stdout в случае ошибки
        sys.stdout = old_stdout
        print(f"Ошибка при выполнении анализа: {e}")
        raise

def create_html_report(console_output, result_df, contingency_table, percentage_table, 
                      cluster_summary_df, cluster_means, subclass_cluster_map, 
                      n_clusters, output_html_path):
    """Создание HTML отчета с результатами анализа"""
    
    # Создаем стилизованный HTML
    html_content = f"""
    <!DOCTYPE html>
    <html lang="ru">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Анализ кластеризации с подклассами</title>
        <style>
            body {{
                font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
                line-height: 1.6;
                margin: 0;
                padding: 20px;
                background-color: #ffffff; /* БЕЛЫЙ ФОН */
            }}
            .container {{
                max-width: 1200px;
                margin: 0 auto;
                background: white;
                padding: 30px;
                border-radius: 10px;
                box-shadow: 0 0 10px rgba(0,0,0,0.05); /* Более легкая тень */
            }}
            .header {{
                background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                color: white;
                padding: 30px;
                border-radius: 10px;
                margin-bottom: 30px;
                text-align: center;
            }}
            .section {{
                margin-bottom: 30px;
                padding: 20px;
                background: #ffffff; /* БЕЛЫЙ ФОН СЕКЦИЙ */
                border-radius: 8px;
                border-left: 4px solid #667eea;
                border: 1px solid #e0e0e0; /* Добавляем границу */
            }}
            .section h2 {{
                color: #333;
                border-bottom: 2px solid #667eea;
                padding-bottom: 10px;
                margin-top: 0;
            }}
            .console-output {{
                background: #f8f9fa; /* Светлый фон вместо темного */
                color: #333; /* Темный текст */
                padding: 20px;
                border-radius: 5px;
                font-family: 'Courier New', monospace;
                white-space: pre-wrap;
                overflow-x: auto;
                font-size: 14px;
                border: 1px solid #ddd;
            }}
            table {{
                width: 100%;
                border-collapse: collapse;
                margin: 15px 0;
                background: white;
                border: 1px solid #ddd;
            }}
            table, th, td {{
                border: 1px solid #ddd;
            }}
            th {{
                background: #667eea;
                color: white;
                padding: 12px;
                text-align: left;
            }}
            td {{
                padding: 10px;
                background: white;
            }}
            tr:nth-child(even) {{
                background: #f8f9fa;
            }}
            .highlight {{
                background: #f8f9fa;
                padding: 15px;
                border-radius: 5px;
                border-left: 4px solid #667eea;
            }}
            .summary-grid {{
                display: grid;
                grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
                gap: 15px;
                margin: 20px 0;
            }}
            .summary-card {{
                background: white;
                padding: 15px;
                border-radius: 8px;
                text-align: center;
                box-shadow: 0 2px 5px rgba(0,0,0,0.1);
                border: 1px solid #e0e0e0;
            }}
            .summary-card h3 {{
                margin: 0;
                color: #667eea;
                font-size: 24px;
            }}
            .summary-card p {{
                margin: 5px 0 0 0;
                color: #666;
            }}
            .timestamp {{
                text-align: right;
                color: #888;
                font-style: italic;
                margin-top: 30px;
            }}
            /* Улучшения для таблиц */
            .dataframe {{
                width: 100%;
                border-collapse: collapse;
            }}
            .dataframe th {{
                background: #667eea;
                color: white;
            }}
        </style>
    </head>
    <body>
        <div class="container">
            <div class="header">
                <h1>📊 Анализ кластеризации с подклассами</h1>
                <p>Автоматический отчет по результатам кластеризации K-Means</p>
            </div>
            
            <div class="summary-grid">
                <div class="summary-card">
                    <h3>{len(result_df)}</h3>
                    <p>Всего наблюдений</p>
                </div>
                <div class="summary-card">
                    <h3>{n_clusters}</h3>
                    <p>Кластеров</p>
                </div>
                <div class="summary-card">
                    <h3>{len(result_df.index.get_level_values(1).unique())}</h3>
                    <p>Уникальных подклассов</p>
                </div>
                <div class="summary-card">
                    <h3>{result_df.select_dtypes(include=[np.number]).columns.nunique()}</h3>
                    <p>Признаков</p>
                </div>
            </div>
            
            <div class="section">
                <h2>📋 Полный вывод анализа</h2>
                <div class="console-output">{console_output}</div>
            </div>
            
            <div class="section">
                <h2>🔍 Таблица сопряженности: Подклассы × Кластеры</h2>
                {contingency_table.to_html(classes='dataframe', border=0, index=True)}
            </div>
            
            <div class="section">
                <h2>📈 Процентное распределение подклассов</h2>
                <p><em>Проценты показывают распределение каждого подкласса по кластерам</em></p>
                {percentage_table.round(1).to_html(classes='dataframe', border=0, index=True)}
            </div>
            
            <div class="section">
                <h2>🎯 Анализ чистоты кластеров</h2>
                {cluster_summary_df.to_html(classes='dataframe', border=0, index=False)}
            </div>
            
            <div class="section">
                <h2>📊 Характеристики кластеров</h2>
                <p><em>Средние значения признаков по кластерам</em></p>
                {cluster_means.round(3).to_html(classes='dataframe', border=0, index=True)}
            </div>
            
            <div class="section">
                <h2>🗺️ Маппинг подклассов на кластеры</h2>
                <div class="highlight">
    """
    
    # Добавляем маппинг подклассов
    for subclass, cluster in subclass_cluster_map.items():
        html_content += f"<p><strong>{subclass}</strong> → Основной кластер <strong>{cluster}</strong></p>\n"
    
    html_content += f"""
                </div>
            </div>
            
            <div class="timestamp">
                Отчет сгенерирован: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
            </div>
        </div>
    </body>
    </html>
    """
    
    # Сохраняем HTML файл
    with open(output_html_path, 'w', encoding='utf-8') as f:
        f.write(html_content)

def analyze_cluster_characteristics(result_df, n_clusters):
    """Анализ характеристик кластеров"""
    print(f"\nХАРАКТЕРИСТИКИ КЛАСТЕРОВ (средние значения):")
    print("-" * 60)
    
    numeric_columns = result_df.select_dtypes(include=[np.number]).columns
    numeric_columns = numeric_columns[numeric_columns != 'cluster']  # Исключаем колонку с кластерами
    
    cluster_means = result_df.groupby('cluster')[numeric_columns].mean()
    print(cluster_means.round(3))

def get_subclass_cluster_mapping(result_df):
    """Получить маппинг подклассов на основные кластеры"""
    subclass_cluster_map = {}
    
    for subclass in result_df.index.get_level_values(1).unique():
        subclass_data = result_df.xs(subclass, level=1)
        if len(subclass_data) > 0:
            # Определяем основной кластер для подкласса
            main_cluster = subclass_data['cluster'].mode()
            if len(main_cluster) > 0:
                subclass_cluster_map[subclass] = main_cluster[0]
    
    return subclass_cluster_map

In [27]:
def random_forest_analysis_with_subclasses(df, target_column=None, problem_type='auto', 
                                         test_size=0.2, random_state=42, 
                                         output_html_path='random_forest_analysis_report.html',
                                         n_estimators=100, max_depth=None):
    """
    Анализ данных с помощью Random Forest и анализ подклассов
    с выводом результатов в HTML файл
    """
    
    # Перехватываем вывод в консоль
    old_stdout = sys.stdout
    sys.stdout = captured_output = StringIO()
    
    try:
        print("=" * 70)
        print("АНАЛИЗ С ПОМОЩЬЮ RANDOM FOREST")
        print("=" * 70)
        
        # Шаг 1: Подготовка данных
        print("1. Подготовка данных...")
        
        # Определяем целевую переменную
        if target_column is None:
            # Используем второй уровень индекса как целевую переменную
            target = df.index.get_level_values(1)
            features_df = df.reset_index(drop=True)
            target_name = "Второй уровень индекса"  # Исправлено с "Первый уровень индекса"
        else:
            target = df[target_column]
            features_df = df.drop(columns=[target_column])
            target_name = target_column
        
        # Определяем тип задачи
        if problem_type == 'auto':
            if target.dtype == 'object' or target.nunique() < 20:
                problem_type = 'classification'
            else:
                problem_type = 'regression'
        
        print(f"   Тип задачи: {problem_type.upper()}")
        print(f"   Целевая переменная: {target_name}")
        print(f"   Уникальных значений в целевой переменной: {target.nunique()}")
        
        # Кодируем целевую переменную для классификации
        if problem_type == 'classification':
            le = LabelEncoder()
            target_encoded = le.fit_transform(target)
            class_names = le.classes_
            print(f"   Классы: {list(class_names)}")
            print(f"   Количество классов: {len(class_names)}")
        else:
            target_encoded = target
            le = None
            class_names = None
        
        # Шаг 2: Предобработка признаков
        print("2. Предобработка признаков...")
        
        # Удаляем нечисловые колонки если есть
        numeric_columns = features_df.select_dtypes(include=[np.number]).columns
        features_numeric = features_df[numeric_columns].copy()
        
        # Проверяем наличие пропущенных значений
        if features_numeric.isnull().sum().sum() > 0:
            missing_count = features_numeric.isnull().sum().sum()
            print(f"   Обнаружено пропущенных значений: {missing_count}")
            features_numeric = features_numeric.fillna(features_numeric.mean())
        
        # Масштабирование признаков
        scaler = StandardScaler()
        features_scaled = scaler.fit_transform(features_numeric)
        
        print(f"   Размерность данных: {features_scaled.shape}")
        print(f"   Количество признаков: {features_scaled.shape[1]}")
        
        # Шаг 3: Разделение на train/test с контролем стратификации
        print("3. Разделение на обучающую и тестовую выборки...")
        
        # Для стратификации убедимся, что все классы присутствуют в обеих выборках
        if problem_type == 'classification':
            # Проверяем распределение классов перед разделением
            unique_classes, class_counts = np.unique(target_encoded, return_counts=True)
            print(f"   Распределение классов перед разделением:")
            for cls, count in zip(unique_classes, class_counts):
                print(f"      Класс {cls} ({class_names[cls]}): {count} samples")
            
            # Используем стратификацию только если все классы имеют достаточное количество образцов
            min_samples_per_class = class_counts.min()
            if min_samples_per_class < 2:
                print("   Предупреждение: некоторые классы имеют менее 2 образцов, стратификация отключена")
                stratify = None
            else:
                stratify = target_encoded
        else:
            stratify = None
        
        X_train, X_test, y_train, y_test, indices_train, indices_test = train_test_split(
            features_scaled, target_encoded, features_df.index, 
            test_size=test_size, random_state=random_state, stratify=stratify
        )
        
        print(f"   Обучающая выборка: {X_train.shape[0]} наблюдений")
        print(f"   Тестовая выборка: {X_test.shape[0]} наблюдений")
        
        # Проверяем наличие всех классов в обучающей и тестовой выборках
        if problem_type == 'classification':
            train_classes = np.unique(y_train)
            test_classes = np.unique(y_test)
            print(f"   Классы в обучающей выборке: {len(train_classes)}")
            print(f"   Классы в тестовой выборке: {len(test_classes)}")
            
            # Если в тестовой выборке не все классы, используем только присутствующие
            if len(test_classes) < len(class_names):
                print("   Предупреждение: не все классы присутствуют в тестовой выборке")
                # Создаем маску для присутствующих классов
                present_classes_mask = np.isin(np.arange(len(class_names)), test_classes)
                present_class_names = class_names[present_classes_mask]
                print(f"   Используемые классы для отчета: {list(present_class_names)}")
            else:
                present_class_names = class_names
        
        # Шаг 4: Обучение Random Forest
        print("4. Обучение Random Forest...")
        
        if problem_type == 'classification':
            rf_model = RandomForestClassifier(
                n_estimators=n_estimators,
                max_depth=max_depth,
                random_state=random_state,
                n_jobs=-1,
                class_weight='balanced'  # Добавляем балансировку классов
            )
        else:
            rf_model = RandomForestRegressor(
                n_estimators=n_estimators,
                max_depth=max_depth,
                random_state=random_state,
                n_jobs=-1
            )
        
        rf_model.fit(X_train, y_train)
        
        # Шаг 5: Предсказания и оценка модели
        print("5. Оценка модели...")
        
        y_pred = rf_model.predict(X_test)
        
        # Метрики в зависимости от типа задачи
        if problem_type == 'classification':
            accuracy = accuracy_score(y_test, y_pred)
            print(f"   Accuracy: {accuracy:.4f}")
            print(f"\n   Classification Report:")
            print("   " + "-" * 50)
            
            # Используем только классы, присутствующие в тестовой выборке
            try:
                clf_report = classification_report(
                    y_test, y_pred, 
                    target_names=present_class_names, 
                    output_dict=True,
                    zero_division=0
                )
                clf_report_df = pd.DataFrame(clf_report).transpose()
                print(clf_report_df.to_string(float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
            except ValueError as e:
                print(f"   Ошибка при создании classification report: {e}")
                print("   Используем числовые метки классов...")
                clf_report = classification_report(
                    y_test, y_pred, 
                    output_dict=True,
                    zero_division=0
                )
                clf_report_df = pd.DataFrame(clf_report).transpose()
                print(clf_report_df.to_string(float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
        else:
            mse = mean_squared_error(y_test, y_pred)
            mae = mean_absolute_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)
            print(f"   Mean Squared Error: {mse:.4f}")
            print(f"   Mean Absolute Error: {mae:.4f}")
            print(f"   R² Score: {r2:.4f}")
        
        # Остальной код остается без изменений...
        # [Шаги 6-8 остаются такими же как в оригинальной функции]
        
        # Шаг 6: Анализ важности признаков
        print("\n6. Анализ важности признаков...")
        
        feature_importance = pd.DataFrame({
            'feature': numeric_columns,
            'importance': rf_model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        print("   Топ-10 самых важных признаков:")
        for i, row in feature_importance.head(10).iterrows():
            print(f"      {row['feature']}: {row['importance']:.4f}")
        
        # Шаг 7: Анализ подклассов
        print("\n7. Анализ распределения подклассов...")
        
        # Создаем DataFrame с результатами
        results_df = features_df.copy()
        results_df['actual'] = target_encoded
        results_df['predicted'] = rf_model.predict(features_scaled)
        results_df['is_correct'] = (results_df['actual'] == results_df['predicted']) if problem_type == 'classification' else None
        
        # Анализ точности по подклассам
        if problem_type == 'classification' and df.index.nlevels > 1:
            subclass_accuracy = {}
            subclasses = df.index.get_level_values(1).unique()
            
            for subclass in subclasses:
                subclass_mask = df.index.get_level_values(1) == subclass
                if subclass_mask.sum() > 0:
                    subclass_actual = results_df.loc[subclass_mask, 'actual']
                    subclass_pred = results_df.loc[subclass_mask, 'predicted']
                    acc = accuracy_score(subclass_actual, subclass_pred)
                    subclass_accuracy[subclass] = acc
            
            subclass_acc_df = pd.DataFrame.from_dict(subclass_accuracy, orient='index', columns=['Accuracy'])
            subclass_acc_df = subclass_acc_df.sort_values('Accuracy', ascending=False)
            
            print(f"\n   Точность по подклассам:")
            print("   " + "-" * 40)
            for subclass, acc in subclass_acc_df.head(10).iterrows():
                print(f"      {subclass}: {acc['Accuracy']:.4f}")
        
        # Получаем весь вывод
        console_output = captured_output.getvalue()
        
        # Восстанавливаем stdout
        sys.stdout = old_stdout
        
        # Создаем HTML отчет
        create_rf_html_report(console_output, results_df, feature_importance, 
                             rf_model, problem_type, class_names, le,
                             X_test, y_test, y_pred, output_html_path)
        
        print(f"\n✅ HTML отчет сохранен в файл: {output_html_path}")
        
        return (results_df, rf_model, scaler, feature_importance, 
                X_train, X_test, y_train, y_test, le)
        
    except Exception as e:
        # Восстанавливаем stdout в случае ошибки
        sys.stdout = old_stdout
        print(f"Ошибка при выполнении анализа: {e}")
        raise

def create_rf_html_report(console_output, results_df, feature_importance, 
                         rf_model, problem_type, class_names, label_encoder,
                         X_test, y_test, y_pred, output_html_path):
    """Создание HTML отчета с результатами анализа Random Forest"""
    
    # Создаем визуализации
    plt.style.use('default')
    
    # График важности признаков
    fig1, ax1 = plt.subplots(figsize=(10, 6))
    top_features = feature_importance.head(15)
    ax1.barh(top_features['feature'], top_features['importance'])
    ax1.set_xlabel('Важность признака')
    ax1.set_title('Топ-15 самых важных признаков')
    ax1.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    
    # Сохраняем график в base64
    from io import BytesIO
    import base64
    
    buffer1 = BytesIO()
    fig1.savefig(buffer1, format='png', dpi=100, bbox_inches='tight')
    buffer1.seek(0)
    feature_importance_plot = base64.b64encode(buffer1.getvalue()).decode()
    plt.close(fig1)
    
    # Матрица ошибок для классификации
    confusion_matrix_plot = ""
    if problem_type == 'classification':
        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(y_test, y_pred)
        fig2, ax2 = plt.subplots(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                   xticklabels=class_names, yticklabels=class_names)
        ax2.set_xlabel('Предсказанные')
        ax2.set_ylabel('Фактические')
        ax2.set_title('Матрица ошибок')
        plt.tight_layout()
        
        buffer2 = BytesIO()
        fig2.savefig(buffer2, format='png', dpi=100, bbox_inches='tight')
        buffer2.seek(0)
        confusion_matrix_plot = base64.b64encode(buffer2.getvalue()).decode()
        plt.close(fig2)
    
    # Создаем стилизованный HTML
    html_content = f"""
    <!DOCTYPE html>
    <html lang="ru">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Анализ Random Forest</title>
        <style>
            body {{
                font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
                line-height: 1.6;
                margin: 0;
                padding: 20px;
                background-color: #ffffff;
            }}
            .container {{
                max-width: 1400px;
                margin: 0 auto;
                background: white;
                padding: 30px;
                border-radius: 10px;
                box-shadow: 0 0 10px rgba(0,0,0,0.05);
            }}
            .header {{
                background: linear-gradient(135deg, #28a745 0%, #20c997 100%);
                color: white;
                padding: 30px;
                border-radius: 10px;
                margin-bottom: 30px;
                text-align: center;
            }}
            .section {{
                margin-bottom: 30px;
                padding: 20px;
                background: #ffffff;
                border-radius: 8px;
                border-left: 4px solid #28a745;
                border: 1px solid #e0e0e0;
            }}
            .section h2 {{
                color: #333;
                border-bottom: 2px solid #28a745;
                padding-bottom: 10px;
                margin-top: 0;
            }}
            .console-output {{
                background: #f8f9fa;
                color: #333;
                padding: 20px;
                border-radius: 5px;
                font-family: 'Courier New', monospace;
                white-space: pre-wrap;
                overflow-x: auto;
                font-size: 14px;
                border: 1px solid #ddd;
            }}
            table {{
                width: 100%;
                border-collapse: collapse;
                margin: 15px 0;
                background: white;
                border: 1px solid #ddd;
            }}
            table, th, td {{
                border: 1px solid #ddd;
            }}
            th {{
                background: #28a745;
                color: white;
                padding: 12px;
                text-align: left;
            }}
            td {{
                padding: 10px;
                background: white;
            }}
            tr:nth-child(even) {{
                background: #f8f9fa;
            }}
            .highlight {{
                background: #f8f9fa;
                padding: 15px;
                border-radius: 5px;
                border-left: 4px solid #28a745;
            }}
            .summary-grid {{
                display: grid;
                grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
                gap: 15px;
                margin: 20px 0;
            }}
            .summary-card {{
                background: white;
                padding: 15px;
                border-radius: 8px;
                text-align: center;
                box-shadow: 0 2px 5px rgba(0,0,0,0.1);
                border: 1px solid #e0e0e0;
            }}
            .summary-card h3 {{
                margin: 0;
                color: #28a745;
                font-size: 24px;
            }}
            .summary-card p {{
                margin: 5px 0 0 0;
                color: #666;
            }}
            .plot-container {{
                text-align: center;
                margin: 20px 0;
            }}
            .plot-container img {{
                max-width: 100%;
                height: auto;
                border: 1px solid #ddd;
                border-radius: 5px;
            }}
            .timestamp {{
                text-align: right;
                color: #888;
                font-style: italic;
                margin-top: 30px;
            }}
            .model-info {{
                background: #e8f5e8;
                padding: 15px;
                border-radius: 5px;
                margin: 15px 0;
            }}
        </style>
    </head>
    <body>
        <div class="container">
            <div class="header">
                <h1>🌲 Анализ Random Forest</h1>
                <p>Автоматический отчет по результатам анализа с помощью Random Forest</p>
            </div>
            
            <div class="summary-grid">
                <div class="summary-card">
                    <h3>{len(results_df)}</h3>
                    <p>Всего наблюдений</p>
                </div>
                <div class="summary-card">
                    <h3>{len(feature_importance)}</h3>
                    <p>Признаков</p>
                </div>
                <div class="summary-card">
                    <h3>{rf_model.n_estimators}</h3>
                    <p>Деревьев в лесу</p>
                </div>
                <div class="summary-card">
                    <h3>{problem_type.upper()}</h3>
                    <p>Тип задачи</p>
                </div>
            </div>
            
            <div class="model-info">
                <h3>📊 Информация о модели</h3>
                <p><strong>Тип модели:</strong> Random Forest {'Classifier' if problem_type == 'classification' else 'Regressor'}</p>
                <p><strong>Количество деревьев:</strong> {rf_model.n_estimators}</p>
                <p><strong>Максимальная глубина:</strong> {rf_model.max_depth if rf_model.max_depth else 'Не ограничена'}</p>
                <p><strong>Количество признаков:</strong> {rf_model.n_features_in_}</p>
            </div>
            
            <div class="section">
                <h2>📋 Полный вывод анализа</h2>
                <div class="console-output">{console_output}</div>
            </div>
            
            <div class="section">
                <h2>🔍 Важность признаков</h2>
                <p><em>Топ-15 самых важных признаков по версии Random Forest</em></p>
                <div class="plot-container">
                    <img src="data:image/png;base64,{feature_importance_plot}" alt="Feature Importance">
                </div>
                {feature_importance.head(20).to_html(classes='dataframe', border=0, index=False)}
            </div>
    """
    
    # Добавляем матрицу ошибок для классификации
    if problem_type == 'classification' and confusion_matrix_plot:
        html_content += f"""
            <div class="section">
                <h2>🎯 Матрица ошибок</h2>
                <p><em>Визуализация правильных и неправильных предсказаний</em></p>
                <div class="plot-container">
                    <img src="data:image/png;base64,{confusion_matrix_plot}" alt="Confusion Matrix">
                </div>
            </div>
        """
    
    html_content += f"""
            <div class="section">
                <h2>📈 Детальная информация о признаках</h2>
                <p><em>Полная таблица важности всех признаков</em></p>
                {feature_importance.to_html(classes='dataframe', border=0, index=False)}
            </div>
            
            <div class="timestamp">
                Отчет сгенерирован: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
            </div>
        </div>
    </body>
    </html>
    """
    
    # Сохраняем HTML файл
    with open(output_html_path, 'w', encoding='utf-8') as f:
        f.write(html_content)

# ДОПОЛНИТЕЛЬНЫЕ ФУНКЦИИ ДЛЯ УГЛУБЛЕННОГО АНАЛИЗА

def analyze_feature_relationships(rf_model, feature_names, top_n=10):
    """Анализ взаимосвязей признаков"""
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("Топ-{} самых важных признаков:".format(top_n))
    for i, row in importance_df.head(top_n).iterrows():
        print(f"  {row['feature']}: {row['importance']:.4f}")
    
    return importance_df

In [123]:
from itertools import combinations

def get_all_combinations_as_lists(lst):
    """Возвращает комбинации в виде списков"""
    result = []
    for r in range(1, len(lst) + 1):
        for combo in combinations(lst, r):
            result.append(list(combo))
    return result

In [126]:
feature_columns = ['E2E3', 'C_org', 'Suva', 'pH', 'eH', 'Asm_350', 'Asm_280',
       'Component_1']
combinations_list = get_all_combinations_as_lists(feature_columns)
len(combinations_list)

255

In [128]:
for combination in combinations_list:

    results_df, rf_model, scaler, feature_importance, X_train, X_test, y_train, y_test, le = random_forest_analysis_with_subclasses(
        MainData[combination],
        problem_type='classification',
        test_size=0.2,
        output_html_path=str(Path.joinpath(Path.cwd().parent,"Server","Received data","Clustering",f"Rforest_f-{combination}.html"))
    )

    for num_cluster in [2,3]:

        result_df, kmeans, scaler, contingency_table, percentage_table, subclass_cluster_map = cluster_analysis_with_subclasses(
            MainData[combination],
            n_clusters=num_cluster,
            output_html_path=Path.joinpath(Path.cwd().parent,"Server","Received data","Clustering",f"KMeans_f-{combination}_c-{num_cluster}.html"))


✅ HTML отчет сохранен в файл: c:\Users\mnbv2\Desktop\lab\KNP analysis\Server\Received data\Clustering\Rforest_f-['E2E3'].html

✅ HTML отчет сохранен в файл: c:\Users\mnbv2\Desktop\lab\KNP analysis\Server\Received data\Clustering\KMeans_f-['E2E3']_c-2.html

✅ HTML отчет сохранен в файл: c:\Users\mnbv2\Desktop\lab\KNP analysis\Server\Received data\Clustering\KMeans_f-['E2E3']_c-3.html

✅ HTML отчет сохранен в файл: c:\Users\mnbv2\Desktop\lab\KNP analysis\Server\Received data\Clustering\Rforest_f-['C_org'].html

✅ HTML отчет сохранен в файл: c:\Users\mnbv2\Desktop\lab\KNP analysis\Server\Received data\Clustering\KMeans_f-['C_org']_c-2.html

✅ HTML отчет сохранен в файл: c:\Users\mnbv2\Desktop\lab\KNP analysis\Server\Received data\Clustering\KMeans_f-['C_org']_c-3.html

✅ HTML отчет сохранен в файл: c:\Users\mnbv2\Desktop\lab\KNP analysis\Server\Received data\Clustering\Rforest_f-['Suva'].html

✅ HTML отчет сохранен в файл: c:\Users\mnbv2\Desktop\lab\KNP analysis\Server\Received data\Clus